In [ ]:
import pandas as pd
import numpy as np
from joblib import parallel_backend
import json
import os
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, GroupKFold, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import datetime  # Added for timestamps
import time
warnings.filterwarnings('ignore')

# ==========================================
# 0. CONFIGURATION & DIRECTORIES
# ==========================================
DATA_FILE = "clean_data.csv"
ID_COL = "filename"
TARGET_COL = "uptake(mmol/g) methane at 65 bar"
TOPO_COL = "Crystalnet"

DIRS = [
    "data_processed/split_definitions", 
    "results/models", "results/metrics", "results/predictions", 
    "results/tables", "manuscript_assets", "supplementary_assets"
]
for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)

# ==========================================
# 1. DATA ACQUISITION & CORE PREPARATION
# ==========================================
def load_and_track_data(filepath):
    print("Executing Data Integration & Filtering...")
    df_raw = pd.read_csv(filepath)#.sample(1000)
    tracker = {"Rows initially loaded": len(df_raw)}
    
    # Target Selection Metrics (SI Table Prep)
    methane_cols = [c for c in df_raw.columns if 'methane' in c.lower()]
    target_stats = df_raw[methane_cols].describe().T
    target_stats['missing_fraction'] = df_raw[methane_cols].isna().mean()
    target_stats.to_csv("results/tables/target_selection_metrics.csv")
    
    # 1. Drop missing targets
    df = df_raw.dropna(subset=[TARGET_COL])
    tracker["Intersection after target drop"] = len(df)
    
    # 2. Drop duplicates
    df = df.drop_duplicates(subset=[ID_COL])
    tracker["Rows after dropping duplicate IDs"] = len(df)
    
    # 3. Conservative filtering (No negative physics)
    df = df[(df['Di'] >= 0) & (df['Df'] >= 0) & (df['Density'] >= 0) & (df['ASA'] >= 0)]
    tracker["Final core benchmark set (valid physics)"] = len(df)
    
    pd.DataFrame(list(tracker.items()), columns=["Processing Stage", "Count"]).to_csv("results/tables/Table3_Data_Integration.csv", index=False)
    
    # Topology Handling & Frequency Grouping (Methodology Section 26)
    topo_counts = df[TOPO_COL].value_counts(dropna=False)
    top_topos = set(topo_counts.nlargest(20).index)
    rare_labels = set(topo_counts[topo_counts < 50].index)
    
    df['topology_label'] = df[TOPO_COL].apply(lambda x: x if x in top_topos else 'other')
    df['topology_frequency_group'] = df[TOPO_COL].apply(lambda x: "rare" if x in rare_labels else "common")
    
    topo_counts.to_csv("results/tables/topology_frequency_groups.csv")
    return df.reset_index(drop=True)

# ==========================================
# 2. DEFINING DESCRIPTOR FAMILIES
# ==========================================
def feature_engineering(df):
    df = df.copy()
    eps = 1e-8
    # Enriched Interpretable Features (incorporating your custom columns where applicable)
    df['lcd_pld_ratio'] = df['Di'] / (df['Df'] + eps)
    df['cavity_window_gap'] = df['Dif'] 
    df['sa_pv_ratio'] = df['ASA'] / (df['AVA'] + eps)
    df['vf_density_ratio'] = df['AVAf'] / (df['Density'] + eps)
    df['log_pld_plus1'] = np.log1p(df['Df'].clip(lower=0))
    df['log_lcd_plus1'] = np.log1p(df['Di'].clip(lower=0))
    return df

# Family A
GEOMETRY_COLS = ['Density', 'AVAf', 'AVA', 'ASA', 'Df', 'Di']
# Family B (Custom + Engineered)
ENRICHED_COLS = GEOMETRY_COLS + [
    'UC_volume', 'vASA', 'NASA', 'POAVA', 
    'lcd_pld_ratio', 'cavity_window_gap', 'sa_pv_ratio', 'vf_density_ratio', 'log_pld_plus1', 'log_lcd_plus1'
]
# Family C
TOPO_CAT_COLS = ['topology_label']

FAMILIES = {
    "geometry_only": {"num": GEOMETRY_COLS, "cat": []},
    "enriched_interpretable": {"num": ENRICHED_COLS, "cat": []},
    "topology_only": {"num": [], "cat": TOPO_CAT_COLS},
    "geometry_plus_topology": {"num": ENRICHED_COLS, "cat": TOPO_CAT_COLS}
}

# ==========================================
# 3. SPLITTING PROTOCOL
# ==========================================
def make_splits(df, n_splits=5):
    out_dir = "data_processed/split_definitions"
    for seed in range(n_splits):
        tr, te = train_test_split(df.index, test_size=0.2, random_state=seed)
        with open(f"{out_dir}/random_split_seed_{seed}.json", "w") as f:
            json.dump({"seed": seed, "train_ids": df.loc[tr, ID_COL].tolist(), "test_ids": df.loc[te, ID_COL].tolist()}, f)
            
    gkf = GroupKFold(n_splits=5)
    for i, (tr, te) in enumerate(gkf.split(df, groups=df['topology_label'])):
        with open(f"{out_dir}/group_split_fold_{i}.json", "w") as f:
            json.dump({"fold": i, "train_ids": df.loc[tr, ID_COL].tolist(), "test_ids": df.loc[te, ID_COL].tolist()}, f)

# ==========================================
# 4. MODEL SETUP & HYPERPARAMETER OPTIMIZATION
# ==========================================
PARAM_GRIDS = {
    "ridge": {"model__alpha": [0.1, 1.0, 10.0, 100.0]},
    "rf": {
        "model__n_estimators": [25, 50, 100],
        "model__max_depth": [3, 5, 10],
        "model__min_samples_leaf": [1, 3, 10]
    },
    "hgb": {
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [None, 8, 16],
        "model__max_iter": [100, 200]
    },
    "mlp": {
        "model__hidden_layer_sizes": [(32,), (64,), (32, 32)],
        "model__alpha": [1e-4, 1e-3, 1e-2],
        "model__learning_rate_init": [1e-3, 5e-4]
    }
}


def build_pipeline(model_name, num_cols, cat_cols):
    scale = model_name in ["ridge", "mlp"]
    transformers = []
    if num_cols:
        steps = [("imputer", SimpleImputer(strategy="median"))]
        if scale: steps.append(("scaler", StandardScaler()))
        transformers.append(("num", Pipeline(steps), num_cols))
    if cat_cols:
        # FIX: Added sparse=False to OneHotEncoder
        transformers.append(("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")), 
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))  # <-- ADDED sparse=False
        ]), cat_cols))

        
    prep = ColumnTransformer(transformers)
    models = {
        "ridge": Ridge(random_state=42),
        "rf": RandomForestRegressor(n_jobs=-1, random_state=42,verbose=1),
        "hgb": HistGradientBoostingRegressor(random_state=42,verbose=1),
        "mlp": MLPRegressor(max_iter=100, early_stopping=True, random_state=42,verbose=1)
    }
    return Pipeline([("preprocessor", prep), ("model", models[model_name])])

def top_fraction_overlap(y_true, y_pred, frac=0.1):
    k = max(1, int(frac * len(y_true)))
    true_top = set(pd.Series(y_true).sort_values(ascending=False).head(k).index)
    pred_top = set(pd.Series(y_pred).sort_values(ascending=False).head(k).index)
    return len(true_top.intersection(pred_top)) / k

def run_evaluation(df, split_prefix, out_csv):
    print(f"\n{'='*60}")
    print(f"STARTING EVALUATION: {split_prefix}")
    print(f"Started at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*60}")
    
    split_files = sorted(Path("data_processed/split_definitions").glob(f"{split_prefix}_*.json"))
    metrics_list, preds_list = [], []
    inner_cv = KFold(n_splits=3, shuffle=True, random_state=0)
    
    # --- PROGRESS TRACKING LOGIC ---
    total_splits = len(split_files)
    models_list = ["ridge", "rf", "hgb", "mlp"]
    
    # Calculate total valid combinations for the progress percentage
    valid_combos_per_split = 0
    for fam in FAMILIES.keys():
        for mod in models_list:
            if fam == 'topology_only' and mod in ['ridge', 'mlp']: continue
            valid_combos_per_split += 1
    
    total_runs = total_splits * valid_combos_per_split
    global_run_counter = 0
    start_time_all = time.time()
    # -------------------------------

    for split_idx, sf in enumerate(split_files):
        with open(sf, "r") as f: payload = json.load(f)
        identifier = payload.get("seed", payload.get("fold"))
        
        train_df = df[df[ID_COL].isin(set(payload["train_ids"]))]
        test_df = df[df[ID_COL].isin(set(payload["test_ids"]))]
        
        print(f"\n>>> SPLIT/FOLD {split_idx + 1} of {total_splits} (ID: {identifier})")
        
        for fam_idx, (fam, cols) in enumerate(FAMILIES.items()):
            print(f"    Descriptor Family {fam_idx + 1}/{len(FAMILIES)}: {fam}")
            
            for mod in models_list:
                if fam == 'topology_only' and mod in ['ridge', 'mlp']:
                    continue 
                
                global_run_counter += 1
                progress_pct = (global_run_counter / total_runs) * 100
                timestamp = datetime.datetime.now().strftime("%H:%M:%S")
                
                print(f"    [{timestamp}] Progress: {progress_pct:.1f}% | Run {global_run_counter}/{total_runs}")
                print(f"    Training Model: {mod}...", end=" ", flush=True)
                
                run_start = time.time()
                
                pipe = build_pipeline(mod, cols["num"], cols["cat"])
                all_feats = cols["num"] + cols["cat"]
                
                # GRID SEARCH
                # Note: verbose=0 here to keep your console clean, 
                # because the print statements above give you the "control"
                grid = GridSearchCV(pipe, PARAM_GRIDS[mod], scoring="r2", cv=inner_cv, n_jobs=-1, refit=True, verbose=1)
                
                with parallel_backend('threading'):
                    grid.fit(train_df[all_feats], train_df[TARGET_COL])
                
                preds = grid.predict(test_df[all_feats])
                
                run_duration = time.time() - run_start
                print(f"Finished in {run_duration:.1f}s")
                
                metrics_list.append({
                    "split_id": identifier, "family": fam, "model": mod,
                    "r2": r2_score(test_df[TARGET_COL], preds),
                    "mae": mean_absolute_error(test_df[TARGET_COL], preds),
                    "rmse": np.sqrt(mean_squared_error(test_df[TARGET_COL], preds)),
                    "spearman": spearmanr(test_df[TARGET_COL], preds).statistic,
                    "top10_overlap": top_fraction_overlap(test_df[TARGET_COL].values, preds)
                })
                
                preds_list.append(pd.DataFrame({ID_COL: test_df[ID_COL], "y_true": test_df[TARGET_COL], "y_pred": preds, "split_id": identifier, "family": fam, "model": mod}))
                
    total_duration = time.time() - start_time_all
    print(f"\n{'='*60}")
    print(f"COMPLETED {split_prefix}")
    print(f"Total time elapsed: {datetime.timedelta(seconds=int(total_duration))}")
    print(f"{'='*60}\n")
    
    # Save results [rest of your saving logic remains the same]
    pd.DataFrame(metrics_list).to_csv(f"results/metrics/{out_csv}_raw.csv", index=False)
    pd.concat(preds_list).to_csv(f"results/predictions/{out_csv}_preds.csv", index=False)


# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    df = load_and_track_data(DATA_FILE)
    df_eng = feature_engineering(df)
    
    make_splits(df_eng)
    
    run_evaluation(df_eng, "random_split", "Table5_random")
    run_evaluation(df_eng, "group_split", "Table6_groupaware")
    df_eng.to_csv('df_eng.csv')
    